# Pipeline: PDF to Feature Extraction

Complete pipeline:
1. Parse PDF with Docling
2. Extract features with LLM (comparison)
3. Extract features with FAISS embeddings (main approach)
4. Compare results

## Step 1: Environment Setup

In [18]:
import sys
import numpy as np
import os

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"NumPy location: {np.__file__}")

# Force CPU usage
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import torch
torch.set_default_device('cpu')

Python executable: /Users/marielouise/miniconda3/envs/sentiment_env/bin/python
Python version: 3.10.17 | packaged by conda-forge | (main, Apr 10 2025, 22:15:49) [Clang 18.1.8 ]
NumPy version: 1.26.4
NumPy location: /Users/marielouise/miniconda3/envs/sentiment_env/lib/python3.10/site-packages/numpy/__init__.py


## Step 2: Parse PDF with Docling

In [19]:
# Verify PDF exists
import os
from pathlib import Path

source_pdf = "/Users/marielouise/CSF-Webapp/CSF_Report_USZ_Example_A.pdf"

print(f"File exists: {os.path.exists(source_pdf)}")
print(f"File is readable: {os.access(source_pdf, os.R_OK)}")
print(f"File size: {os.path.getsize(source_pdf)} bytes")

# Verify it's a valid PDF
with open(source_pdf, 'rb') as f:
    header = f.read(4)
    print(f"PDF header: {header}")
    print(f"Is valid PDF: {header == b'%PDF'}")

File exists: True
File is readable: True
File size: 338229 bytes
PDF header: b'%PDF'
Is valid PDF: True


In [20]:
# Parse PDF with Docling
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
import time

print("Parsing PDF with Docling...")

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = True
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4, 
    device=AcceleratorDevice.CPU
)

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend
        )
    }
)

start_time = time.time()
result = doc_converter.convert(source_pdf)
elapsed = time.time() - start_time

print(f"Done: PDF parsed in {elapsed:.2f}s")

# Save outputs, only using markdown now
markdown_text = result.document.export_to_markdown()
# json_output = result.document.model_dump_json(indent=2)

with open('markdown.md', 'w', encoding='utf-8') as f:
    f.write(markdown_text)

# with open('output.json', 'w', encoding='utf-8') as f:
#     f.write(json_output)


Parsing PDF with Docling...


2025-11-14 11:15:38,462 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-14 11:15:38,477 - INFO - Going to convert document batch...
2025-11-14 11:15:38,481 - INFO - Initializing pipeline for StandardPdfPipeline with options hash c209dd628446177ed34ad72bc155d23b
2025-11-14 11:15:38,485 - INFO - Accelerator device: 'cpu'
2025-11-14 11:15:40,378 - INFO - Accelerator device: 'cpu'
2025-11-14 11:15:41,430 - INFO - Processing document CSF_Report_USZ_Example_A.pdf


len(pages)=1, 0-0
len(valid_pages)=1
len(valid_page_images)=1
len(pages)=1, 1-1
len(valid_pages)=1
len(valid_page_images)=1


2025-11-14 11:15:57,815 - INFO - Finished converting document CSF_Report_USZ_Example_A.pdf in 19.37 sec.


Done: PDF parsed in 19.40s


## Step 3: Extract Features with LLM Prompt 1

In [ ]:
import os
os.environ['OLLAMA_NUM_GPU'] = '0'  # Force CPU only
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

import ollama
import time
import json
from synonym_dict import FEATURE_SYNONYMS

print("="*70)
print("LLM EXTRACTION WITH SYNONYMS")
print("="*70)

# Load documents
with open('markdown.md', 'r', encoding='utf-8') as f:
    markdown_text = f.read()

def create_llm_prompt(document_text, feature_synonyms):
    expected_output = {feature: None for feature in feature_synonyms.keys()}
    
    # Show key synonyms for clarity
    synonym_examples = []
    for feature_name, languages in list(feature_synonyms.items())[:10]:
        de_syns = languages.get('de', [])[:2]
        if de_syns:
            synonym_examples.append(f'"{feature_name}": also "{de_syns[0]}"')
    
    prompt = f"""Extract medical lab values from this document. Output ONLY valid JSON.

CRITICAL: Output format must be FLAT - just "field": "value with unit" (NOT nested objects!)

CORRECT FORMAT:
"Albumin (Serum)": "77.2 g/l"
"IgG (Liquor)": "49.70 mg/l"

WRONG FORMAT (DO NOT USE):
"Albumin (Serum)": {{"Wert": "77.2", "Unit": "g/l"}}

Use these EXACT field names: {', '.join(list(feature_synonyms.keys())[:15])}... and {len(feature_synonyms)-15} more.

DOCUMENT:
{document_text}

Output ONLY flat JSON with "field": "value unit" or "field": null
"""
    return prompt

prompt = create_llm_prompt(markdown_text, FEATURE_SYNONYMS)

print(f"Running extraction with llama3.2")
start_time = time.time()

# Pull the model like this: ollama pull llama3.2:3b
response = ollama.chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': prompt}],
    options={
        'temperature': 0, 
        'num_ctx': 8192, # maybe too big for the small model
        'num_thread': 4  # Limit CPU threads
    }
)

elapsed_time = time.time() - start_time
print(f"Completed in {elapsed_time:.2f}s")

# Parse response
response_text = response['message']['content']

if '```json' in response_text:
    response_text = response_text.split('```json')[1].split('```')[0]
elif '```' in response_text:
    response_text = response_text.split('```')[1].split('```')[0]

try:
    extracted_json = json.loads(response_text.strip())
    extracted_count = sum(1 for v in extracted_json.values() if v and v != "null")
    total_features = len(FEATURE_SYNONYMS)
    
    print(f"Extracted: {extracted_count}/{total_features} ({extracted_count/total_features*100:.1f}%)")
    
    # Save
    with open('llm_extracted.json', 'w', encoding='utf-8') as f:
        json.dump(extracted_json, f, indent=2, ensure_ascii=False)
    
    llm_summary = {
        'model': 'llama3.2:3b',
        'extraction_time': elapsed_time,
        'extracted_count': extracted_count,
        'accuracy': extracted_count/total_features*100
    }
    
    with open('llm_extraction_summary.json', 'w', encoding='utf-8') as f:
        json.dump(llm_summary, f, indent=2)
    
except json.JSONDecodeError as e:
    print(f"Failed to parse: {e}")
    with open('llm_raw_response.txt', 'w') as f:
        f.write(response_text)

LLM EXTRACTION WITH SYNONYMS
Running extraction with llama3.2


2025-11-14 11:18:57,808 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Completed in 138.59s
Extracted: 26/41 (63.4%)


## Step 3: Extract Features with LLM Prompt 2

In [ ]:
import os
os.environ['OLLAMA_NUM_GPU'] = '0'  # Force CPU only
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

import ollama
import time
import json
#from synonym_dict import FEATURE_SYNONYMS

print("="*70)
print("LLM Extraction without synonyms")
print("="*70)

# Load documents
with open('markdown.md', 'r', encoding='utf-8') as f:
    markdown_text = f.read()

def create_llm_prompt(document_text, feature_synonyms):
    expected_output = {feature: None for feature in feature_synonyms.keys()}
    
    # Show key synonyms for clarity
    synonym_examples = []
    for feature_name, languages in list(feature_synonyms.items())[:10]:
        de_syns = languages.get('de', [])[:2]
        if de_syns:
            synonym_examples.append(f'"{feature_name}": also "{de_syns[0]}"')
    
    expected_features = [
        "Aktivierte Lymphozyten", 
"Albumin (Serum)", 
"Albumin-Quotient", 
"Basophile", 
"Erythrozyten", 
"Glukose (Liquor)", 
"Glukose (Plasma)", 
"Glukose-Quotient", 
"Grobdifferenzierung", 
"IgA (Liquor)", 
"IgA (Serum)", 
"IgA(LOC)", 
"IgA-Quotient", 
"IgG (Liquor)", 
"IgG (Serum)", 
"IgG(LOC)", 
"IgG-Index", 
"IgG-Quotient", 
"IgM (Liquor)", 
"IgM (Serum)", 
"IgM(IF)", 
"IgM(LOC)", 
"IgM-Quotient", 
"Kappa (Liquor)", 
"Kappa (Serum)", 
"Kappa(IF)", 
"Kappa(LOC)", 
"Kappa-Index", 
"Kappa-Quotient", 
"Korrigierte Zellzahl", 
"Laktat (Liquor)", 
"Lambda (Liquor)", 
"Lambda (Serum)", 
"Lambda-Quotient", 
"Leukophagen (bei Durchsicht)", 
"Lymphozyten", 
"MRZ(H)-Beurteilung", 
"Makrophagen (bei Durchsicht)",
"Makrophagen/Monozyten", 
"Maligne Zellen", 
"Neutrophile", 
"OKB-Beurteilung", 
"Oligoklonale Banden (Liquor)", 
"Oligoklonale Banden (Serum)", 
"Plasmazellen", 
"Plasmazellen (bei Durchsicht)", 
"Totalprotein (Liquor)", 
"Zellzahl", 
"GESCHLECHT", 
"ALTER_J"

    ]

    prompt = f"""Extract medical lab values from this document. Output ONLY valid JSON.

CRITICAL: Output format must be FLAT - just "field": "value with unit" (NOT nested objects!)

CORRECT FORMAT:
"Albumin (Serum)": "77.2 g/l"
"IgG (Liquor)": "49.70 mg/l"

WRONG FORMAT (DO NOT USE):
"Albumin (Serum)": {{"Wert": "77.2", "Unit": "g/l"}}

Use these EXACT field names: {expected_features}. We call them expected features. If you don't find a feature corresponding to one of these field names, don't hallucinate it, but output null as the value
for that feature. The document does not contain all features so this is expected. Use the context to understand which fields correspond to the expected feature names. There can be synonyms, 
and some values may be duplicate. Only provide one value for each of the expected features. 
If there are values that are text based, use the exact text. For example if it says "nicht bestimmt" , write exactly that as the value for that feature, not null.

DOCUMENT:
{document_text}

Output ONLY flat JSON with "field": "value unit" or "field": null. Only output a valid JSON!
"""
    return prompt

prompt = create_llm_prompt(markdown_text, FEATURE_SYNONYMS)

print(f"Running extraction with llama3.2")
start_time = time.time()

# Pull the model like this: ollama pull llama3.2:3b
## Now: I tried llama3.2:1b instead of llama3.2:3b on the better prompt

response = ollama.chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': prompt}],
    options={
        'temperature': 0, 
        'num_ctx': 8192,
        'num_thread': 4  # Limit CPU threads
    }
)

elapsed_time = time.time() - start_time
print(f"Completed in {elapsed_time:.2f}s")

# Parse response
response_text = response['message']['content']

if '```json' in response_text:
    response_text = response_text.split('```json')[1].split('```')[0]
elif '```' in response_text:
    response_text = response_text.split('```')[1].split('```')[0]

try:
    extracted_json = json.loads(response_text.strip())
    extracted_count = sum(1 for v in extracted_json.values() if v)
    total_features = len(FEATURE_SYNONYMS)
    
    print(f"Extracted: {extracted_count}/{total_features} ({extracted_count/total_features*100:.1f}%)")
    
    # Save
    with open('llm_extracted_prompt2_3B.json', 'w', encoding='utf-8') as f:
        json.dump(extracted_json, f, indent=2, ensure_ascii=False)
    
    llm_summary = {
        'model': 'llama3.2:3b',
        'extraction_time': elapsed_time,
        'extracted_count': extracted_count,
        'accuracy': extracted_count/total_features*100
    }
    
    with open('llm_extraction_summary_prompt2_3B.json', 'w', encoding='utf-8') as f:
        json.dump(llm_summary, f, indent=2)
    
except json.JSONDecodeError as e:
    print(f"Failed to parse: {e}")
    with open('llm_raw_response_prompt2_3B.txt', 'w') as f:
        f.write(response_text)

LLM EXTRACTION WITHOUT SYNONYMS
Running extraction with llama3.2


2025-11-14 14:11:01,140 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Completed in 157.04s
Extracted: 32/41 (78.0%)


## Step 4: Extract Features with FAISS Embeddings (Main Approach)

In [24]:
# Parse markdown to extract label-value pairs
import re
import json

with open('markdown.md', 'r', encoding='utf-8') as f:
    markdown_text = f.read()

def extract_from_markdown(markdown_text):
    lines = markdown_text.split('\n')
    extracted_pairs = []
    
    for line in lines:
        if '|' not in line or line.strip().startswith('|-'):
            continue
        
        cells = [cell.strip() for cell in line.split('|')]
        cells = [c for c in cells if c]
        
        if len(cells) < 2:
            continue
        
        label = cells[0]
        
        if any(kw in label.lower() for kw in ['routineprogramm', 'bemerkungen', 'referenzbereich']):
            continue
        
        value = None
        unit = None
        
        for i in range(1, len(cells)):
            cell = cells[i]
            if cell in ['*', '', '-'] or cell.startswith('<') or cell.startswith('>'):
                continue
            
            if re.search(r'\d', cell) or cell.lower() in ['m', 'w', 'männlich', 'weiblich']:
                value = cell
                if len(cells) > i + 1:
                    potential_unit = cells[-1]
                    if potential_unit and potential_unit not in ['*', ''] and len(potential_unit) < 20:
                        unit = potential_unit
                break
        
        if label and value:
            extracted_pairs.append({
                'label': label,
                'value': value,
                'unit': unit if unit else ''
            })
    
    return extracted_pairs

extracted_pairs = extract_from_markdown(markdown_text)
for pair in extracted_pairs:
    print(f"{pair['label']}: {pair['value']} {pair['unit']}")

Material vom:: 20.03.2025 XXXXXXXXX
Gesamtzellzahl: 39 /μl
Lymphozyten: 81.0 %
Aktivierte Lymphozyten: 0.9 %
Plasmazellen: 3.0 %
Monozyten: 7.5 %
Totalprotein im Liquor: 574 mg/l
Albumin im Liquor: 391 mg/l
Albumin im Serum: 77.2 g/l
Albumin - Quotient: 18.1 × 10 ⁻ ³
IgG im Liquor: 49.70 mg/l
IgG im Serum: 11.80 g/l
IgG-Quotient: 14.2 × 10 ⁻ ³
IgG-Index: 0.57 < 0.57
IgA im Liquor: 17.90 mg/l
IgA im Serum: 2.93 g/l
IgA-Quotient: 0.4 × 10 ⁻ ³
IgM im Liquor: 0.337 mg/l
IgM im Serum: 0.441 g/l
IgM-Quotient: 3.0 × 10 ⁻ ³
Kappa im Liquor: 0.756 mg/l
Kappa im Serum: 9.100 mg/l
Kappa-Quotient: 41.733 × 10 ⁻ ³
Kappa-Index: 2.523 < 9.58
Lambda im Liquor: 0.716 mg/l
Lambda im Serum: 12.900 mg/l
Lambda - Quotient: 21.827 × 10 ⁻ ³
Lambda - Index: 2.559 < 5.85
IgG-IF: 3.16 %
IgA-IF: 1.72 %
IgM-IF: 2.14 %
Kappa-IF: 41 %
Oligoklonale Banden im Liquor: positiv: 3 Banden negativ
Oligoklonale Banden im Serum: positiv: 3 Banden negativ
Masern Antikörper-Index: 1.01 < 1.5
Röteln Antikörper-Index: 0.95 < 1.

In [ ]:
import time
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from synonym_dict import FEATURE_SYNONYMS


# uncomment the name of the model you want to use 
#model_name = "thenlper/gte-small"
#model_name = "BAAI/bge-base-en-v1.5"
#model_name = "BAAI/bge-large-en-v1.5"
#model_name = "Fjoralb1/multilingual-e5-small-nli-matryoshka-128"
#model_name = "paraphrase-multilingual-MiniLM-L12-v2"
model_name = "mixedbread-ai/mxbai-embed-large-v1"

model = SentenceTransformer(model_name)

start_build = time.time()

feature_embeddings_dict = {}
feature_names = []

for feature_name, languages in FEATURE_SYNONYMS.items():
    synonyms = []
    for lang_syns in languages.values():
        synonyms.extend(lang_syns)
    emb = model.encode(synonyms, show_progress_bar=False)
    feature_embeddings_dict[feature_name] = np.mean(emb, axis=0)
    feature_names.append(feature_name)

embedding_dimension = len(next(iter(feature_embeddings_dict.values())))
matrix = np.vstack([feature_embeddings_dict[n] for n in feature_names]).astype("float32")

faiss.normalize_L2(matrix)

index = faiss.IndexFlatIP(embedding_dimension)
index.add(matrix)

elapsed = time.time() - start_build

safe_name = model_name.replace("/", "_")
faiss_filename = f"faiss_feature_index_{safe_name}.bin"
meta_filename = f"faiss_metadata_{safe_name}.pkl"

faiss.write_index(index, faiss_filename)

metadata = {
    "feature_names": feature_names,
    "embedding_dimension": embedding_dimension,
    "model_name": model_name,
    "build_time_seconds": elapsed
}

with open(meta_filename, "wb") as f:
    pickle.dump(metadata, f)

print(f"FAISS index saved as: {faiss_filename}")
print(f"Metadata saved as:   {meta_filename}")
print(f"Features: {len(feature_names)}, Dimension: {embedding_dimension}, Build time: {elapsed:.2f}s")

2025-11-14 15:56:32,528 - INFO - Use pytorch device_name: mps
2025-11-14 15:56:32,529 - INFO - Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1
2025-11-14 15:57:08,972 - INFO - 1 prompt is loaded, with the key: query


FAISS index saved as: faiss_feature_index_mixedbread-ai_mxbai-embed-large-v1.bin
Metadata saved as:   faiss_metadata_mixedbread-ai_mxbai-embed-large-v1.pkl
Features: 41, Dimension: 1024, Build time: 8.79s


In [ ]:
# Per-Document Extraction (Run for each new document on docling output)
import time
import numpy as np
import faiss
import pickle
import json
from sentence_transformers import SentenceTransformer

print("="*70)
print("FAISS EXTRACTION (PER-DOCUMENT)")
print("="*70)

# uncomment model name below to test it (has to be the same one as in the box above!!!)

#model_name = "thenlper/gte-small" # dim: 384
#model_name = "BAAI/bge-base-en-v1.5"
#model_name = "BAAI/bge-large-en-v1.5"
#model_name = "Fjoralb1/multilingual-e5-small-nli-matryoshka-128"
#model_name = "paraphrase-multilingual-MiniLM-L12-v2" # 384 dimensions
model_name = "mixedbread-ai/mxbai-embed-large-v1" # 1024 dim


safe_name = model_name.replace("/", "_")
faiss_filename = f"faiss_feature_index_{safe_name}.bin"
meta_filename = f"faiss_metadata_{safe_name}.pkl"

# Load pre-built FAISS index
start_load = time.time()

index = faiss.read_index(faiss_filename)

with open(meta_filename, "rb") as f:
    metadata = pickle.load(f)

feature_names = metadata["feature_names"]
embedding_dimension = metadata["embedding_dimension"]

# Load embedding model
model = SentenceTransformer(metadata["model_name"])

elapsed_load = time.time() - start_load
print(f"Loaded index in {elapsed_load:.3f}s, Features: {len(feature_names)}, Dimension: {embedding_dimension}")

# Extract labels from document (using previously extracted pairs)
start_encode = time.time()

# extracted_pairs were created in the previous cell
labels = [pair["label"] for pair in extracted_pairs]

# Encode labels (THIS IS THE PER-DOCUMENT COST)
label_embeddings = model.encode(labels, show_progress_bar=False).astype("float32")
faiss.normalize_L2(label_embeddings)

elapsed_encode = time.time() - start_encode
print(f"Encoded {len(labels)} labels in {elapsed_encode:.3f}s")

# Search FAISS index
start_search = time.time()

k = 1  # Top 1 match
similarities, indices = index.search(label_embeddings, k)

elapsed_search = time.time() - start_search
print(f"Searched {len(labels)} queries in {elapsed_search:.3f}s")

# Build matches
matches = []
for i, (similarity, idx) in enumerate(zip(similarities[:, 0], indices[:, 0])):
    matches.append({
        "original_label": extracted_pairs[i]["label"],
        "matched_feature": feature_names[idx],
        "confidence": float(similarity),
        "value": extracted_pairs[i]["value"],
        "unit": extracted_pairs[i]["unit"]
    })

# Apply threshold
CONFIDENCE_THRESHOLD = 0.70
extracted_features = {}
high_confidence = 0

for match in matches:
    if match["confidence"] >= CONFIDENCE_THRESHOLD:
        value_with_unit = f"{match['value']} {match['unit']}".strip()
        extracted_features[match["matched_feature"]] = value_with_unit
        high_confidence += 1

# Fill missing features
for feature in feature_names:
    if feature not in extracted_features:
        extracted_features[feature] = None

# Calculate total time breakdown
total_per_doc_time = elapsed_load + elapsed_encode + elapsed_search

print("\n" + "="*70)
print("TIMING")
print("="*70)
print("PER-DOCUMENT COST:")
print(f"1.Load index: {elapsed_load:.3f}s")
print(f"2.Encode labels: {elapsed_encode:.3f}s : LIVE EMBEDDING COST")
print(f"3.Search FAISS: {elapsed_search:.3f}s")
print("---")
print(f"Total per doc: {total_per_doc_time:.3f}s")

print("RESULTS:")
print(f"Extracted: {high_confidence}/{len(feature_names)} ({high_confidence/len(feature_names)*100:.1f}%)")

# Save results (model-specific filenames)
results_filename = f"faiss_extracted_{safe_name}.json"
summary_filename = f"faiss_extraction_summary_{safe_name}.json"

with open(results_filename, "w", encoding="utf-8") as f:
    json.dump(extracted_features, f, indent=2, ensure_ascii=False)

faiss_summary = {
    "one_time_build": metadata.get("build_time", metadata.get("build_time_seconds")),
    "per_doc_load_time": elapsed_load,
    "per_doc_encode_time": elapsed_encode,  # THIS IS THE LIVE EMBEDDING COST
    "per_doc_search_time": elapsed_search,
    "total_per_doc_time": total_per_doc_time,
    "extracted_count": high_confidence,
    "accuracy": high_confidence / len(feature_names) * 100
}

with open(summary_filename, "w", encoding="utf-8") as f:
    json.dump(faiss_summary, f, indent=2)


2025-11-14 15:57:44,913 - INFO - Use pytorch device_name: mps
2025-11-14 15:57:44,914 - INFO - Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1


FAISS EXTRACTION (PER-DOCUMENT)


2025-11-14 15:57:59,206 - INFO - 1 prompt is loaded, with the key: query


Loaded index in 14.304s, Features: 41, Dimension: 1024
Encoded 38 labels in 2.245s
Searched 38 queries in 0.011s

TIMING
PER-DOCUMENT COST:
1.Load index: 14.304s
2.Encode labels: 2.245s : LIVE EMBEDDING COST
3.Search FAISS: 0.011s
---
Total per doc: 16.561s
RESULTS:
Extracted: 34/41 (82.9%)


In [ ]:
from sentence_transformers import SentenceTransformer

model1 = SentenceTransformer('intfloat/multilingual-e5-small')
model2 = SentenceTransformer('all-MiniLM-L6-v2')
# this was just a test, to see if the model sizes fit what i expected them to be
print(f"Model 1 parameters: {sum(p.numel() for p in model1.parameters()):,}")
print(f"Model 2 parameters: {sum(p.numel() for p in model2.parameters()):,}")
print(f"Model 1 layers: {model1[0].auto_model.config.num_hidden_layers}")
print(f"Model 2 layers: {model2[0].auto_model.config.num_hidden_layers}")

2025-11-14 11:29:35,039 - INFO - Use pytorch device_name: mps
2025-11-14 11:29:35,040 - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-small
2025-11-14 11:29:58,593 - INFO - Use pytorch device_name: mps
2025-11-14 11:29:58,596 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Model 1 parameters: 117,653,760
Model 2 parameters: 22,713,216
Model 1 layers: 12
Model 2 layers: 6


## Step 5: Compare Results to ground truth

In [86]:
import json
import os

# Load ground truth
with open('ground_truth.json', 'r') as f:
    ground_truth = json.load(f)

# Methods:
methods = [
    ("LLM first prompt", "llm_extracted.json", "llm_extraction_summary.json"),
      ("LLM second prompt, 3B", "llm_extracted_prompt2_3B.json", "llm_extraction_summary_prompt2_3B.json"),
 ("FAISS using Fjoralb1",
     "faiss_extracted_Fjoralb1_multilingual-e5-small-nli-matryoshka-128.json",
     "faiss_extraction_summary_Fjoralb1_multilingual-e5-small-nli-matryoshka-128.json"),

    ("FAISS using MiniLM",  "faiss_extracted_paraphrase-multilingual-MiniLM-L12-v2.json",
     "faiss_extraction_summary_paraphrase-multilingual-MiniLM-L12-v2.json"),

    ("FAISS using gte-small",
     "faiss_extracted_thenlper_gte-small.json",
     "faiss_extraction_summary_thenlper_gte-small.json"),
     
     ("FAISS using BAAI_base",
     "faiss_extracted_BAAI_bge-base-en-v1.5.json",
     "faiss_extraction_summary_BAAI_bge-base-en-v1.5.json"),

    ("FAISS using mxbai",
     "faiss_extracted_mixedbread-ai_mxbai-embed-large-v1.json",
     "faiss_extraction_summary_mixedbread-ai_mxbai-embed-large-v1.json"),
     ("FAISS using BAAI_large",
     "faiss_extracted_BAAI_bge-large-en-v1.5.json",
     "faiss_extraction_summary_BAAI_bge-large-en-v1.5.json"),
]

with open("evaluation_results_new.txt", "w") as out:
    out.write("Model evaluation results\n")
    out.write("=" * 70 + "\n\n")

    for method_name, pred_file, summary_file in methods:
        out.write(f"{method_name}:\n")

        with open(pred_file, "r") as f:
            predictions = json.load(f)

        correct = 0
        total = len(ground_truth)

        # Compare predictions to GT
        for feature, true_value in ground_truth.items():
            pred_value = predictions.get(feature, None)

            true_norm = str(true_value).strip() if true_value not in [None, "null", ""] else None
            pred_norm = str(pred_value).strip() if pred_value not in [None, "null", ""] else None

            if true_norm == pred_norm:
                correct += 1
            else:
                out.write(f"  {feature}: expected '{true_value}', got '{pred_value}'\n")
        out.write(f"Correct: {correct}/{total}")

       # load timing info from the summary files 
        if summary_file and os.path.exists(summary_file):
            with open(summary_file, "r") as f:
                summary = json.load(f)

            build = summary.get("one_time_build", None)
            load  = summary.get("per_doc_load_time", None)
            enc   = summary.get("per_doc_encode_time", None)
            srch  = summary.get("per_doc_search_time", None)
            total_time = summary.get("total_per_doc_time", None)
            llm_time = summary.get("extraction_time", None)

            out.write(f"\nBuild index: {build:.3f}\n" if build is not None else "")
            out.write(f"Load index:{load:.3f}\n"  if load  is not None else "")
            out.write(f"Encode labels: {enc:.3f}\n"  if enc   is not None else "")
            out.write(f"Search index:{srch:.3f}\n" if srch  is not None else "")
            out.write(f"Total per doc:{total_time:.3f}\n" if total_time is not None else "")
            out.write(f"LLM time: {llm_time:.3f}\n" if llm_time is not None else "")

        out.write("\n" + "-" * 70 + "\n\n")